[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/225/blob/main/08_data_cleaning/08_4_String_Cleaning.ipynb)

# 08.4: String Cleaning

Text columns are the messiest part of most real datasets. A groupby that should produce two rows ("male" and "female") silently produces five because of capitalization, trailing spaces, and inconsistent spelling. An extract that should pull a title out of a name fails because the regex doesn't account for a missing period.

This notebook covers the pandas string methods that handle these problems: case normalization, whitespace stripping, search and replace, containment checks, and prefix/suffix matching. All of them work through the `.str` accessor, which applies a string operation to every element of a Series. Let's load the data.

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_titanic.csv"
df = pd.read_csv(url)
df.columns = ["survived", "pclass", "name", "sex", "age", "sibsp", "parch", "fare"]
df[["name", "sex"]].head()

,name,sex
0,Mr. Owen Harris Braund,male
1,Mrs. John Bradley (Florence Briggs Thayer) Cum...,female
2,Miss. Laina Heikkinen,female
3,Mrs. Jacques Heath (Lily May Peel) Futrelle,female
4,Mr. William Henry Allen,male


The setup shows name and sex for the first five rows. Names follow a `Title. Lastname, Firstname` pattern that will be useful to parse. The sex column looks clean here, but real entry-by-hand data rarely stays that way.

## Why string cleaning matters

Suppose you want to count passengers by sex. You already know how to do that with `value_counts()`. But what happens to that count if the data was entered by hand and some rows say `"Female"` with a capital F, and others have a trailing space?

In [2]:
# Simulate formatting inconsistencies in the sex column
sex_messy = df["sex"].copy()
sex_messy.iloc[1] = "Female"    # capitalized
sex_messy.iloc[3] = " female"   # leading space
sex_messy.iloc[5] = "male "     # trailing space

sex_messy.value_counts()

sex
male       572
female     312
Female       1
 female      1
male         1
Name: count, dtype: int64

Three entries that are logically identical to `"female"` or `"male"` are now counted as separate categories. A `groupby` on this column would split them into four groups instead of two. No error is raised. The problem is invisible unless you look at `value_counts()` carefully.

Two lines fix it.

In [3]:
sex_clean = sex_messy.str.strip().str.lower()
sex_clean.value_counts()

sex
male      573
female    314
Name: count, dtype: int64

`.str.strip()` removed the leading and trailing spaces. `.str.lower()` unified the capitalization. The chain runs left to right: `strip()` runs first, `lower()` runs on the result. The two groups are now cleanly separated.

## Case normalization

`.str.lower()`, `.str.upper()`, and `.str.title()` change the case of every character in a column. `lower()` is the most useful for cleaning because it makes any case comparison work regardless of how the original data was entered.

In [4]:
sample = pd.Series(["New York", "CHICAGO", "los angeles", "San Francisco"])

print("lower: ", sample.str.lower().tolist())
print("upper: ", sample.str.upper().tolist())
print("title: ", sample.str.title().tolist())

lower:  ['new york', 'chicago', 'los angeles', 'san francisco']
upper:  ['NEW YORK', 'CHICAGO', 'LOS ANGELES', 'SAN FRANCISCO']
title:  ['New York', 'Chicago', 'Los Angeles', 'San Francisco']


`title()` capitalizes the first letter of each word, which is useful for display. `lower()` is more reliable for comparisons and grouping because it produces a single canonical form regardless of the original capitalization.

A practical rule: convert to lowercase before any `groupby`, `value_counts()`, or equality check on a string column that was entered by hand.

## Whitespace removal

Whitespace at the start or end of a string is invisible but consequential. `"female"` and `" female"` look identical in a printout but are different strings. `.str.strip()` removes both leading and trailing whitespace. `.str.lstrip()` removes only leading whitespace; `.str.rstrip()` removes only trailing whitespace.

In [5]:
values = pd.Series(["  hello  ", "  world", "foo  ", "bar"])

print("strip: ", values.str.strip().tolist())
print("lstrip:", values.str.lstrip().tolist())
print("rstrip:", values.str.rstrip().tolist())

strip:  ['hello', 'world', 'foo', 'bar']
lstrip: ['hello  ', 'world', 'foo  ', 'bar']
rstrip: ['  hello', '  world', 'foo', 'bar']


`strip()` is the right default for most cleaning tasks. Include it whenever you are loading data from a CSV that was created or exported by humans: spreadsheets and hand-entered CSVs frequently have stray spaces.

## Replacement: `.str.replace()`

`.str.replace(old, new)` replaces all occurrences of a substring within each element. The simplest use case is normalizing inconsistent labels.

In [6]:
# Simulate inconsistent class labels
classes = pd.Series(["1st class", "First class", "1st Class", "2nd class", "3rd class"])
print("Before:", classes.tolist())

# Normalize the first-class entries
cleaned = classes.str.lower().str.replace("first ", "1st ", regex=False)
print("After: ", cleaned.tolist())

Before: ['1st class', 'First class', '1st Class', '2nd class', '3rd class']
After:  ['1st class', '1st class', '1st class', '2nd class', '3rd class']


The `regex=False` argument tells pandas to treat the first argument as a plain string rather than a regular expression pattern. This is safer when your replacement string contains characters that have special meaning in regex (like `.` or `*`). When you want to use pattern matching for more complex replacements, you would set `regex=True`, which is covered in notebook 08.6.

## Filtering rows: `.str.contains()`

`.str.contains(pattern)` returns a boolean Series: `True` where the pattern appears in the string, `False` where it does not. This gives you a mask you can use to filter rows.

In [7]:
# Find all passengers whose name contains 'Mrs'
married_women = df[df["name"].str.contains("Mrs", na=False)]
print(f"Passengers with 'Mrs' in name: {len(married_women)}")
married_women[["name", "survived", "pclass"]].head(5)

Passengers with 'Mrs' in name: 125


,name,survived,pclass
1,Mrs. John Bradley (Florence Briggs Thayer) Cum...,1,1
3,Mrs. Jacques Heath (Lily May Peel) Futrelle,1,1
8,Mrs. Oscar W (Elisabeth Vilhelmina Berg) Johnson,1,3
9,Mrs. Nicholas (Adele Achem) Nasser,1,2
15,Mrs. (Mary D Kingcome) Hewlett,1,2


The `na=False` argument prevents errors when a row has a null value in the name column: nulls evaluate to `False` rather than raising a `TypeError`.

`.str.startswith()` and `.str.endswith()` do the same thing but only check the beginning or end of the string. These are useful for filtering prefixes (country codes, product category codes) or file extensions.

In [8]:
# Show the first few names that start with 'Miss'
miss_mask = df["name"].str.startswith("Miss", na=False)
print(f"Passengers whose name starts with 'Miss': {miss_mask.sum()}")
df.loc[miss_mask, "name"].head(5)

Passengers whose name starts with 'Miss': 182


2                   Miss. Laina Heikkinen
10         Miss. Marguerite Rut Sandstrom
11                Miss. Elizabeth Bonnell
14    Miss. Hulda Amanda Adolfina Vestrom
22                     Miss. Anna McGowan
Name: name, dtype: str

`.str.startswith("Miss")` matches only names where "Miss" is at the very beginning, so it targets the title specifically and avoids matching "Miss" embedded mid-string. The `miss_mask.sum()` call counts `True` values in the boolean Series, giving the number of matching rows without creating a separate DataFrame.

## Measuring strings: `.str.len()` and `.str.count()`

`.str.len()` returns the character length of each string in the column. This is useful for spotting outliers: a name that is 3 characters long or 200 characters long is worth investigating.

In [9]:
df["name_len"] = df["name"].str.len()

print("Shortest names:")
print(df.nsmallest(3, "name_len")[["name", "name_len"]])
print("\nLongest names:")
print(df.nlargest(3, "name_len")[["name", "name_len"]])

df = df.drop(columns=["name_len"])

Shortest names:
             name  name_len
689   Mr. Ali Lam        11
822   Mr. Len Lam        11
73   Mr. Lee Bing        12

Longest names:
                                                  name  name_len
305  Mrs. Victor de Satode (Maria Josefa Perez de S...        81
667  Mrs. Thomas William Solomon (Elizabeth Catheri...        60
25   Mrs. Carl Oscar (Selma Augusta Emilia Johansso...        56


The longest names contain parenthetical notes (maiden names or alternate names), which explains their length. The shortest names are simple first-plus-last combinations. No outliers here, but in a dataset where names were entered by hand, very short or very long entries sometimes indicate data entry errors.

`.str.count(pattern)` counts how many times a pattern appears within each string.

In [10]:
# Count how many spaces are in each name (a rough proxy for word count)
space_counts = df["name"].str.count(" ")
print("Space count distribution:")
space_counts.value_counts().sort_index()

Space count distribution:


name
2     325
3     393
4      81
5      46
6      34
7       7
13      1
Name: count, dtype: int64

Most names have two or three spaces (last name, title, first name, possibly middle name). Names with five or more spaces likely have parenthetical notes. This kind of measurement helps you understand the structure of a string column before you try to parse it.

## Chaining operations

Because each `.str` method returns a new Series, you can chain them directly. A complete normalization of a hand-entered column might look like this: strip whitespace, convert to lowercase, and replace a known variant.

In [11]:
# Simulate a messy city column from a hand-entered form
cities = pd.Series([
    "  New York  ",
    "new york",
    "NEW YORK",
    "nyc",
    " Chicago ",
    "chicago",
    "CHICAGO"
])

cities_clean = (
    cities
    .str.strip()
    .str.lower()
    .str.replace("nyc", "new york", regex=False)
)

cities_clean.value_counts()

new york    4
chicago     3
Name: count, dtype: int64

Seven original strings, four distinct raw values, two clean groups after the three-step normalization. This is the basic pattern for any string-cleaning task: strip, lowercase, then handle known variants with `replace()`.

## What's next

The `.str` accessor can normalize and filter strings in a column. But some string columns contain multiple pieces of information packed into one field, such as a full name with title and last name, an address with street and city, or a product code with category and identifier. Notebook 08.5 covers the tools for pulling those pieces apart: `str.split()` and `str.extract()`.